In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F
project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
orders = spark.read.format("delta")\
    .load("abfss://bronze@secondstorage89.dfs.core.windows.net/orders/")


In [0]:
orders.show(10)

In [0]:
orders.printSchema()

In [0]:
orders.groupBy("order_id") \
  .count() \
  .filter("count > 1") \
  .show()

In [0]:
orders.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNull()
).count()

In [0]:
orders.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNull()
).select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(truncate=False)

In [0]:


orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in orders.columns
]).show()

In [0]:
delivered_missing_date = (
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNull()
)

In [0]:
orders.filter(delivered_missing_date).show(truncate=True)

In [0]:
quarantine_orders = orders.filter(delivered_missing_date)

silver_orders = orders.filter(~delivered_missing_date)

In [0]:
silver_orders.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNull()
).count()

In [0]:
silver_orders.filter(
    F.col("order_approved_at").isNotNull() &
    (F.col("order_approved_at") < F.col("order_purchase_timestamp"))
).count()

In [0]:
silver_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    (F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp"))
).count()

In [0]:
silver_orders.filter(
    F.col("order_delivered_customer_date").isNotNull() &
    (F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"))
).count()

purchase → approved → delivery
              ↑
carrier timestamp thoda pehle

In [0]:
silver_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    (F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp"))
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(20, truncate=False)

In [0]:
silver_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    (
        F.col("order_delivered_carrier_date")
        < F.col("order_purchase_timestamp")
    )
).select(
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_carrier_date"
).withColumn(
    "_difference_minutes",
    (
        F.unix_timestamp("order_purchase_timestamp")
        - F.unix_timestamp("order_delivered_carrier_date")
    ) / 60
).describe().show()

In [0]:
silver_orders.filter(
    F.col("order_delivered_carrier_date").isNotNull() &
    (
        F.col("order_delivered_carrier_date")
        < F.col("order_purchase_timestamp")
    )
).select(
    F.col("order_id"),
    F.col("order_purchase_timestamp"),
    F.col("order_delivered_carrier_date")
).withColumn(
    "_difference_minutes",
    (
        F.unix_timestamp("order_purchase_timestamp")
        - F.unix_timestamp("order_delivered_carrier_date")
    ) / 60
).orderBy(
    F.col("_difference_minutes").desc()
).show(20, truncate=False)

In [0]:
silver_orders = silver_orders.withColumn(
    "_carrier_before_purchase",
    (
        F.col("order_delivered_carrier_date").isNotNull() &
        (
            F.col("order_delivered_carrier_date")
            < F.col("order_purchase_timestamp")
        )
    )
)

In [0]:
silver_orders.show(10)

In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://silver@secondstorage89.dfs.core.windows.net/orders/"
    )